# 作業

請使用 Python 完成一個可以做到：
當使用者輸入：

```python
query = "如何提高肌肉量？"
```
 
系統能找出語意最接近的文章標題。

In [2]:
query = "如何增加肌肉量？"

documents = [
    "如何透過重量訓練增加肌肉量",
    "減脂期間的飲食控制技巧",
    "Python 資料分析入門指南",
    "如何改善睡眠品質",
    "提升心肺耐力的運動方式",
    "大型語言模型的工作原理",
    "高蛋白飲食對健身的幫助",
    "時間管理的五個技巧"
]

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 載入 embedding model
embedding_model = SentenceTransformer("intfloat/multilingual-e5-small")

# multilingual-e5-small 的輸出維度
EMBEDDING_DIM = 384
print("Embedding model:", "intfloat/multilingual-e5-small")
print("Embedding dimension:", EMBEDDING_DIM)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model: intfloat/multilingual-e5-small
Embedding dimension: 384


我在作業也是利用 `SentenceTransformers` 的 `intfloat/multilingual-e5-small` 模型將文章標題向量化到 384 維空間。

In [5]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()  # 從 .env 檔案載入環境變數

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

In [3]:
# 將文件轉換為向量
document_vectors = embedding_model.encode(documents, normalize_embeddings=True)
query_vector = embedding_model.encode([f"query: {query}"], normalize_embeddings=True)

# 計算相似度
similarities = cosine_similarity(query_vector, document_vectors).flatten()

# 找出最相似的文本（篩選相似度 top 5 且大於 threshold 的文本）
threshold = 0.35
top_k_indices = similarities.argsort()[-5:][::-1]
retrieved_texts = [documents[idx] for idx in top_k_indices if similarities[idx] > threshold]
print("檢索到的文本:", retrieved_texts)

檢索到的文本: ['如何透過重量訓練增加肌肉量', '提升心肺耐力的運動方式', '高蛋白飲食對健身的幫助', '如何改善睡眠品質', '減脂期間的飲食控制技巧']


我採取的方法是：
- 先取得文章標題的向量表示。
- 再將使用者輸入的 query 向量化。
- 比對使用者輸入的 query 向量與文章標題的向量，取最相關的前 5 個文章標題，然後篩選出大於 0.35 threshold 的文章標題。

我目前觀測這個結果除了 "如何改善睡眠品質" 這個檢索到的文章標題之外，其他的文章標題都與 query 的主題相關。 "如何改善睡眠品質" 這個文章標題似乎與 query 的主題不太相關，我覺得有可算是個 Hard Negative 的例子，因為它的語意與 query 的主題不太相關，但向量距離卻很近。

In [6]:
# 利用檢索到的文本生成回答
d = " ".join(retrieved_texts)
completion = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": f"根據以下上文回答使用者問題，如果上下文沒有提及就回答不知道：{d}"},
        {"role": "user", "content": query}
    ]
)
print("生成的回答:", completion.choices[0].message.content)

生成的回答: 可以透過以下步驟增加肌肉量：

1. **重量訓練**：定期進行重量訓練可以幫助增加肌肉量。要專注於Compound運動，如深蹲、硬拉、bench press和划船等動作，這些運動可以同時訓練多個肌肉群。
2. **逐漸增加負荷**：隨著時間的推移，逐漸增加訓練的重量或阻力，以挑戰肌肉，刺激肌肉生長。
3. **合理的訓練頻率和休息**：每個肌肉群每週訓練3-4次，同時給予足夠的休息時間讓肌肉恢復和生長。
4. **高蛋白飲食**：高蛋白飲食是增加肌肉量的關鍵。每天攝入足夠的蛋白質，以支持肌肉生長和修復。
5. **足夠的睡眠**：充足的睡眠對肌肉恢復和生長至關重要。每晚應該睡眠7-9小時。

增加肌肉量需要時間和耐心，請堅持並有計劃地進行訓練和營養補給。


我目前觀測這個 RAG 系統的效果是是十分滿意的，能夠找到與 query 相關的文章標題，不過仍似乎受到 "如何改善睡眠品質" 這個檢索到的文章標題的影響，導致 LLM 回答提及了要保持足夠睡眠時間的重要性。

---

我在這個作業用的向量化方法是 
- SentenceTransformers 的 intfloat/multilingual-e5-small  
- 向量維度是 384  
- 有做向量正規化

相似度計算的部分是： 
- 利用餘弦相似度
- 先算 query 向量和所有文件向量的相似度，先取 Top 5，然後利用課程教的 threshold 保留分數大於 0.35 的結果

最後找出的結果是：  
- 如何透過重量訓練增加肌肉量  
- 提升心肺耐力的運動方式  
- 高蛋白飲食對健身的幫助  
- 如何改善睡眠品質  
- 減脂期間的飲食控制技巧  

其中最接近的第一名是：如何透過重量訓練增加肌肉量，與詢問的問題非常相關。